# Zebrafish SRA API pull + 5-run test download

Goal: record a reproducible, **idempotent** workflow to (1) fetch run metadata from NCBI SRA and (2) download a small subset of reads for **5 runs** as a quick test.

Everything is organized under this folder:
- `Semester5/BIOL550/group_project/zebrafish/`

Downloaded reads are written under `data/` (gitignored).

## Outline

1. Setup + tool checks
2. Fetch RunInfo via Entrez API (SRA)
3. Create a 5-run “test list” (small runs)
4. Download a small subset of FASTQs for those 5 runs
5. Validate outputs (counts + file sizes)

In [ ]:
from __future__ import annotations

import csv
import gzip
import os
import shutil
import subprocess
from pathlib import Path

# --- Config (edit these if needed) ---
ACC = "PRJNA1277581"               # zebrafish retina regeneration
ORGANISM = "Danio rerio"
N_TEST_RUNS = 5
MAX_SPOTS = 10_000                 # test size per run (spot IDs 1..MAX_SPOTS)
EXCLUDE_RUNS = {"SRR34002423", "SRR34002425"}  # known to sometimes hang on some systems

FORCE_REFETCH_RUNINFO = False      # set True to refetch runinfo even if already present
FORCE_REDOWNLOAD_TEST = False      # set True to redownload test FASTQs even if present

ROOT = Path.cwd()                  # open notebook from zebrafish folder
ZEBRAFISH_ROOT = ROOT
if ZEBRAFISH_ROOT.name != "zebrafish":
    # If the notebook is launched from elsewhere, still locate this repo folder.
    ZEBRAFISH_ROOT = Path("Semester5/BIOL550/group_project/zebrafish").resolve()

SCRIPT_GET_RUNINFO = ZEBRAFISH_ROOT / "scripts" / "get_zebrafish_data_sra.py"
META_DIR = ZEBRAFISH_ROOT / "metadata" / ACC
DATA_TEST_DIR = ZEBRAFISH_ROOT / "data" / "test" / ACC / f"spots_{MAX_SPOTS}"

RUNINFO_CSV = META_DIR / "runinfo.csv"
RUNINFO_FILTERED_CSV = META_DIR / "runinfo.filtered.csv"
RUNS_TEST5_FILE = META_DIR / "runs.test5.smallest_sizeMB.txt"


def which(cmd: str) -> str | None:
    return shutil.which(cmd)


def run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


print("ZEBRAFISH_ROOT:", ZEBRAFISH_ROOT)
print("SCRIPT_GET_RUNINFO exists:", SCRIPT_GET_RUNINFO.exists())
print("esearch:", which("esearch"))
print("efetch:", which("efetch"))
print("fastq-dump:", which("fastq-dump"))
print("prefetch (optional):", which("prefetch"))
print("fasterq-dump (optional):", which("fasterq-dump"))

## 1) Check the workspace structure

This confirms we’re writing metadata and data into the expected subfolders.

In [ ]:
print("META_DIR:", META_DIR)
print("DATA_TEST_DIR:", DATA_TEST_DIR)

ZEBRAFISH_ROOT.mkdir(parents=True, exist_ok=True)
(META_DIR).mkdir(parents=True, exist_ok=True)
(DATA_TEST_DIR).mkdir(parents=True, exist_ok=True)

for rel in ["scripts", "metadata", "notes", "data"]:
    p = ZEBRAFISH_ROOT / rel
    print(rel, "exists=" , p.exists())

## 2) Fetch RunInfo (SRA) via API

We call the existing script, but skip re-fetching if `runinfo.csv` already exists (unless `FORCE_REFETCH_RUNINFO=True`).

In [ ]:
if FORCE_REFETCH_RUNINFO and RUNINFO_CSV.exists():
    RUNINFO_CSV.unlink()

if not RUNINFO_CSV.exists():
    if not SCRIPT_GET_RUNINFO.exists():
        raise FileNotFoundError(SCRIPT_GET_RUNINFO)

    run([
        "python3",
        str(SCRIPT_GET_RUNINFO),
        "--acc",
        ACC,
        "--out-dir",
        str(META_DIR),
        "--organism",
        ORGANISM,
        "--write-download-urls",
    ])
else:
    print("RunInfo already exists, skipping:", RUNINFO_CSV)

# Quick sanity check
rows = list(csv.DictReader(RUNINFO_CSV.read_text(encoding="utf-8").splitlines()))
print("runinfo rows:", len(rows))
print("first Run:", rows[0].get("Run"))

## 3) Create the 5-run test list

We pick the 5 smallest runs (by `size_MB`) from RunInfo, excluding any SRRs in `EXCLUDE_RUNS`.

In [ ]:
rows = list(csv.DictReader(RUNINFO_CSV.read_text(encoding="utf-8").splitlines()))
vals: list[tuple[float, str]] = []
for r in rows:
    srr = (r.get("Run") or "").strip()
    if not srr or srr in EXCLUDE_RUNS:
        continue
    try:
        size = float(r.get("size_MB") or "nan")
    except Exception:
        size = float("nan")
    if size == size:
        vals.append((size, srr))

vals.sort()
selected = [srr for _, srr in vals[:N_TEST_RUNS]]

if len(selected) != N_TEST_RUNS:
    raise RuntimeError(f"Expected {N_TEST_RUNS} runs, got {len(selected)}: {selected}")

RUNS_TEST5_FILE.write_text("".join(f"{s}\n" for s in selected), encoding="utf-8")
print("Wrote:", RUNS_TEST5_FILE)
print("Selected SRRs:\n" + "\n".join(selected))

## 4) Download a small FASTQ subset for those 5 runs

This downloads only spot IDs `1..MAX_SPOTS` for each SRR using `fastq-dump` (quick test; avoids multi‑GB downloads).

In [ ]:
if which("fastq-dump") is None:
    raise RuntimeError("Missing `fastq-dump` in PATH. Install SRA Toolkit on this machine.")

selected = [l.strip() for l in RUNS_TEST5_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]

for srr in selected:
    out_dir = DATA_TEST_DIR / srr
    out_dir.mkdir(parents=True, exist_ok=True)

    f1 = out_dir / f"{srr}_1.fastq.gz"
    f2 = out_dir / f"{srr}_2.fastq.gz"

    if not FORCE_REDOWNLOAD_TEST and f1.exists() and f2.exists():
        print("skip (already downloaded):", srr)
        continue

    # Remove partial outputs if present
    for p in [f1, f2, out_dir / f"{srr}_1.fastq", out_dir / f"{srr}_2.fastq", out_dir / f"{srr}.fastq"]:
        if p.exists():
            p.unlink()

    run([
        "fastq-dump",
        "--split-3",
        "-N",
        "1",
        "-X",
        str(MAX_SPOTS),
        "-O",
        str(out_dir),
        srr,
    ])

    # gzip outputs
    for raw in [out_dir / f"{srr}_1.fastq", out_dir / f"{srr}_2.fastq", out_dir / f"{srr}.fastq"]:
        if raw.exists():
            with raw.open("rb") as fin, gzip.open(str(raw) + ".gz", "wb") as fout:
                shutil.copyfileobj(fin, fout)
            raw.unlink()

print("Done. Test data folder:", DATA_TEST_DIR)

## 5) Validate the outputs (quick checks)

We verify the expected paired FASTQs exist and contain the expected number of reads.

In [ ]:
def fastq_read_count_gz(path: Path) -> int:
    # FASTQ = 4 lines per record
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        lines = sum(1 for _ in f)
    if lines % 4 != 0:
        raise RuntimeError(f"FASTQ line count not divisible by 4: {path} -> {lines}")
    return lines // 4

selected = [l.strip() for l in RUNS_TEST5_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]

for srr in selected:
    out_dir = DATA_TEST_DIR / srr
    f1 = out_dir / f"{srr}_1.fastq.gz"
    f2 = out_dir / f"{srr}_2.fastq.gz"

    if not f1.exists() or not f2.exists():
        raise FileNotFoundError(f"Missing FASTQs for {srr} in {out_dir}")

    n1 = fastq_read_count_gz(f1)
    n2 = fastq_read_count_gz(f2)

    size1 = f1.stat().st_size
    size2 = f2.stat().st_size

    print(f"{srr}: reads1={n1}, reads2={n2}, size1={size1/1e6:.2f}MB, size2={size2/1e6:.2f}MB")

print("All good.")

## Next steps

- If this test looks good, run full downloads on the server (prefer `prefetch` + `fasterq-dump`) and keep the raw data in `data/` (gitignored).
- Keep using `metadata/PRJNA1277581/runinfo.filtered.csv` as the source of truth for which runs are in-scope.